In [11]:
import pandas as pd
from langcodes import Language
import uuid
from datasets import load_dataset, Dataset

In [3]:
seed_df = load_dataset("ljvmiranda921/msde-seed-S1", split="train").to_pandas()
print(seed_df.columns)

Generating train split: 100%|██████████| 449017/449017 [00:00<00:00, 691487.96 examples/s]


Index(['id', 'source', 'language', 'strategy', 'source_id', 'prompt',
       'response'],
      dtype='object')


In [4]:
LANG_MAPPING = {"Tagalog": "tl", "Filipino": "fil"}

Process Cohere Aya Collection

In [8]:
def _iso3_to_iso2(code: str) -> str:
    tag = Language.get(code).to_tag()
    iso2 = tag.split("-")[0]
    return iso2

In [58]:
aya_ds = load_dataset("CohereLabs/aya_collection", "aya_dataset", split="train")
filtered_df = aya_ds.to_pandas()
filtered_df["language"] = filtered_df["language"].apply(_iso3_to_iso2)
filtered_df = filtered_df[filtered_df["language"].isin(set(LANG_MAPPING.values()))].reset_index(drop=True)  # fmt: skip
aya_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(filtered_df))],
        "source": "CohereLabs/aya_collection",
        "language": filtered_df["language"].astype(str).values,
        "prompt": filtered_df["inputs"].astype(str).values,
        "response": filtered_df["targets"].astype(str).values,
        "strategy": [["generate", "respond"] for _ in range(len(filtered_df))],
        "source_id": "CohereLabs/aya_collection_"
        + filtered_df["id"].astype(str).values,
    }
)

Generating train split: 100%|██████████| 202364/202364 [00:00<00:00, 1485697.72 examples/s]


In [35]:
len(aya_df)

1241

Process AllenAI Wildchat

In [36]:
wildchat_df_raw = pd.read_json("data/tl.jsonl.zst", lines=True, compression="zstd")

In [42]:
wildchat_df = pd.DataFrame(
    {
        # fmt: off
        "id": [uuid.uuid4().hex for _ in range(len(wildchat_df_raw))],
        "source": wildchat_df_raw["source"].astype(str).values,
        "language": "tl",
        "prompt": [row[0]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "response": [row[1]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "strategy": [["generate", "respond"] for _ in range(len(wildchat_df_raw))],
        "source_id": "agentlans/allenai-WildChat_" + wildchat_df_raw.index.astype(str),
        # fmt: on
    }
)

Process Saillab Alpaca Training Dataset

In [44]:
alpaca_ds = load_dataset("saillab/alpaca-filipino-cleaned", split="train")

Generating test split: 100%|██████████| 10401/10401 [00:00<00:00, 1350952.43 examples/s]


In [47]:
alpaca_df_raw = alpaca_ds.to_pandas()
# If input is not NaN, append it to the instruction column
alpaca_df_raw["instruction"] = alpaca_df_raw.apply(
    lambda row: (
        row["instruction"]
        if pd.isna(row["input"]) or str(row["input"]).strip().lower() == "nan"
        else f"{row['instruction']}\n\n{row['input']}"
    ),
    axis=1,
)
alpaca_df_raw

,instruction,input,output
0,Gawing wastong pangungusap ang pangungusap na...,Si Peter ay kumain ng mansanas,Si Pedro ay kumain ng mansanas.
1,Bumuo ng isang idyoma na may kaugnayan sa tag...,<walang input>,"""The sky's the limit"" - ibig sabihin ay walang..."
2,Tukuyin ang pinakamalaking lungsod sa Estados...,nan,"Ang pinakamalaking lungsod sa Estados Unidos, ..."
3,Bumuo ng limang positibong pagpapatibay,nan,1. Ikaw ay may kakayahang makamit ang kadakila...
4,Humanap ng 5 salita na naglalarawan sa damdam...,nan,"bigo, bigo, panghinaan ng loob, sama ng loob,..."
...,...,...,...
41596,Mag-imbento ng bagong pagkain na pinagsasama ...,Pineapple at spinach,Ang isang posibleng ulam na pinagsasama ang pi...
41597,Palitan ang pang-uri sa pangungusap ng angkop...,Nagsalita siya ng malakas.,Nagsalita siya ng malakas. (Ang pangungusap a...
41598,"Gamit ang ibinigay na larawan, tukuyin ang ur...",[Larawan],"Ikinalulungkot ko, ngunit ako ay isang modelo..."
41599,Bumuo ng isang dialogue tungkol sa mga kalama...,nan,"User: Hey AI, Pwede ba nating pag-usapan ang p..."


In [48]:
alpaca_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(alpaca_df_raw))],
        "source": "saillab/alpaca-filipino-cleaned",
        "language": "tl",
        "prompt": alpaca_df_raw["instruction"].astype(str).values,
        "response": alpaca_df_raw["output"].astype(str).values,
        "strategy": [["generate", "respond"] for _ in range(len(alpaca_df_raw))],
        "source_id": "saillab/alpaca-filipino-cleaned_"
        + alpaca_df_raw.index.astype(str),
    }
)
alpaca_df

,id,source,language,prompt,response,strategy,source_id
0,dbd9e8087cbb4474bbef05e9e38462e0,saillab/alpaca-filipino-cleaned,tl,Gawing wastong pangungusap ang pangungusap na...,Si Pedro ay kumain ng mansanas.,"[generate, respond]",saillab/alpaca-filipino-cleaned_0
1,385c8ce59fd64136a90d388ae2d9d1cd,saillab/alpaca-filipino-cleaned,tl,Bumuo ng isang idyoma na may kaugnayan sa tag...,"""The sky's the limit"" - ibig sabihin ay walang...","[generate, respond]",saillab/alpaca-filipino-cleaned_1
2,bdbbf2a753c34d2882bb2c2691509997,saillab/alpaca-filipino-cleaned,tl,Tukuyin ang pinakamalaking lungsod sa Estados...,"Ang pinakamalaking lungsod sa Estados Unidos, ...","[generate, respond]",saillab/alpaca-filipino-cleaned_2
3,4ca9d4fa3fd549f4b06b800d560a6054,saillab/alpaca-filipino-cleaned,tl,Bumuo ng limang positibong pagpapatibay,1. Ikaw ay may kakayahang makamit ang kadakila...,"[generate, respond]",saillab/alpaca-filipino-cleaned_3
4,9e641ca3235b4884b3fdf9a617344b7e,saillab/alpaca-filipino-cleaned,tl,Humanap ng 5 salita na naglalarawan sa damdam...,"bigo, bigo, panghinaan ng loob, sama ng loob,...","[generate, respond]",saillab/alpaca-filipino-cleaned_4
...,...,...,...,...,...,...,...
41596,db99939342a14511a1fe62cd6d4df71e,saillab/alpaca-filipino-cleaned,tl,Mag-imbento ng bagong pagkain na pinagsasama ...,Ang isang posibleng ulam na pinagsasama ang pi...,"[generate, respond]",saillab/alpaca-filipino-cleaned_41596
41597,09ea9d6734234f9098ceb05dc10c7965,saillab/alpaca-filipino-cleaned,tl,Palitan ang pang-uri sa pangungusap ng angkop...,Nagsalita siya ng malakas. (Ang pangungusap a...,"[generate, respond]",saillab/alpaca-filipino-cleaned_41597
41598,8b47f7ba65d9474f99a8ac0a28eec5aa,saillab/alpaca-filipino-cleaned,tl,"Gamit ang ibinigay na larawan, tukuyin ang ur...","Ikinalulungkot ko, ngunit ako ay isang modelo...","[generate, respond]",saillab/alpaca-filipino-cleaned_41598
41599,4d2f361efb8b464aa91afb86d7952610,saillab/alpaca-filipino-cleaned,tl,Bumuo ng isang dialogue tungkol sa mga kalama...,"User: Hey AI, Pwede ba nating pag-usapan ang p...","[generate, respond]",saillab/alpaca-filipino-cleaned_41599


In [59]:
tl_datasets = pd.concat(
    [
        wildchat_df,
        alpaca_df,
        aya_df,
    ]
)
tl_datasets["language"] = "tl"
tl_datasets = tl_datasets.reset_index(drop=True)

In [60]:
tl_datasets

,id,source,language,prompt,response,strategy,source_id
0,4a1f9bbbec8046abb9cba87938f8b78c,allenai/WildChat-4.8M,tl,Respond to this message in the appropriate lan...,Ang pag-ibig ay isang napaka-komplikadong emos...,"[generate, respond]",agentlans/allenai-WildChat_0
1,79d3d700871a4ea7af999710c06f32ba,allenai/WildChat-4.8M,tl,anong AI model ka?\nspeak in a more casual tag...,Hey! Ako yung AI assistant mo na laging ready ...,"[generate, respond]",agentlans/allenai-WildChat_1
2,3b7dd25fc6a34054a4688768a3d382be,allenai/WildChat-4.8M,tl,"Ok, sa problem na ito, ang lumalabas na result...",Magandang araw! Mukhang mayroong pagkalito sa ...,"[generate, respond]",agentlans/allenai-WildChat_2
3,d40ae36dbad845149b24d83dbdadf430,allenai/WildChat-4.8M,tl,Ipaliwanag ng maikli kaugnay sa Bana na Espiri...,"Ang ""Bana na Espiritu"" ay isang konsepto na tu...","[generate, respond]",agentlans/allenai-WildChat_3
4,827cba1a9198483697074f4e151f10da,allenai/WildChat-4.8M,tl,Summarize this text in Filipino (make sure to ...,"Ang tekstong ""Ang Kahalagahan ng Pangangalaga ...","[generate, respond]",agentlans/allenai-WildChat_4
...,...,...,...,...,...,...,...
44084,9a912df3a6644cb39d71376aba875e82,CohereLabs/aya_collection,tl,May isang pagkakataon lamang sa iyong buhay na...,Kapag ang iyong anak ay umabot sa edad mo noon...,"[generate, respond]",CohereLabs/aya_collection_201783
44085,67b03e47e4d1428da8b9bbf7b5ac89f7,CohereLabs/aya_collection,tl,Ano ang USB-C?,Ang USB-C ay isang 24-pin USB connector system...,"[generate, respond]",CohereLabs/aya_collection_201987
44086,c2d0199783e2466599f854304d098e61,CohereLabs/aya_collection,tl,"Itama ang mga mali sa talatang ito:\n\n""Umulan...","Ito ang itinamang talata: \n\n""Umulan nang mal...","[generate, respond]",CohereLabs/aya_collection_202001
44087,2d9e2992bcd645b7bf225fddbfea970f,CohereLabs/aya_collection,tl,Si Ann ay may pitong kendi. Binigyan siya ni L...,7 candies + 7 candies = 14 candies.,"[generate, respond]",CohereLabs/aya_collection_202210


In [61]:
seed_df = pd.concat([seed_df, tl_datasets]).reset_index(drop=True)

In [63]:
ds = Dataset.from_pandas(seed_df)
ds.push_to_hub("ljvmiranda921/msde-seed-S1", split="train")

Creating parquet from Arrow format: 100%|██████████| 165/165 [00:01<00:00, 159.68ba/s]
Processing Files (1 / 1): 100%|██████████|  313MB /  313MB, 42.9MB/s  
New Data Upload: 100%|██████████| 28.4MB / 28.4MB, 10.1MB/s  
Creating parquet from Arrow format: 100%|██████████| 165/165 [00:01<00:00, 156.03ba/s]
Processing Files (1 / 1): 100%|██████████|  313MB /  313MB, 28.8MB/s  
New Data Upload: 100%|██████████|  313MB /  313MB, 28.8MB/s  
Creating parquet from Arrow format: 100%|██████████| 165/165 [00:00<00:00, 423.95ba/s]
Processing Files (1 / 1): 100%|██████████|  111MB /  111MB, 22.4MB/s  
New Data Upload: 100%|██████████|  111MB /  111MB, 22.4MB/s  
Uploading the dataset shards: 100%|██████████| 3/3 [00:27<00:00,  9.14s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/ljvmiranda921/msde-seed-S1/commit/be9f4d75817ad9afad1e495571172aaf99ffb91d', commit_message='Upload dataset', commit_description='', oid='be9f4d75817ad9afad1e495571172aaf99ffb91d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ljvmiranda921/msde-seed-S1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ljvmiranda921/msde-seed-S1'), pr_revision=None, pr_num=None)